# SPCE5025 – Spring 2026 Final Exam

## Problem Description

You are an orbit analyst for UltraHuge Aerospace, Inc. Your task is to plan a two-burn Hohmann Transfer sequence for the **ChaserSat** to rendezvous with the **TargetSat**.

> **Note:** The Chaser vehicle is behind the Target and closing. Manage signs so that times come out positive and phase angle changes are consistent with a closing satellite.

**Reference Frame:** True of Date  
**Epoch:** 01 May 2012 00:00:00


## Initial State Vectors

| Component | Target Vehicle | Chaser Vehicle | Units |
|-----------|---------------|----------------|-------|
| X         | -7202235.416  | -6491589.449   | m     |
| Y         |   783685.330  |  2613001.096   | m     |
| Z         |  2839574.411  |  3352419.284   | m     |
| XD        |  -1594.528305 | -3259.157555   | m/s   |
| YD        |  -6621.500276 | -6217.288683   | m/s   |
| ZD        |  -2197.570004 | -1448.482747   | m/s   |


## Part 1: Initial Computations

1. Compute the Keplerian elements for each satellite.
2. Compute the Keplerian orbit period for each satellite.
3. Compute the semi-major axis difference between the target and chaser orbits.
4. Use the semi-major axis difference to estimate the ΔV required to raise the Chaser to the target orbit.
5. Compute the angular difference (phase angle) between the Chaser and Target at the initial epoch.
6. Using the Keplerian periods from Problem 2, compute the phase angle rate between the two satellites.

In [2]:
from standards import *
#initial conditions:
EPOCH = datetime(year=2012, month=5, day=1, hour=0, minute=0, second=0)
POS_TARGET = Vector3(-7202235.416, 783685.330, 2839574.411)
VEL_TARGET = Vector3(-1594.528305, -6621.500276, -2197.570004)
POS_CHASER = Vector3(-6491589.449, 2613001.096, 3352419.284)
VEL_CHASER = Vector3(-3259.157555, -6217.288683, -1448.482747)

#1. Compute the Keplerian elements for each satellite.
KEP_TARGET = KeplerianElements(POS_TARGET, VEL_TARGET)
KEP_CHASER = KeplerianElements(POS_CHASER, VEL_CHASER)
print("Target Kep elements:")
print(f"SMA:  {KEP_TARGET.a:.4f} m")
print(f"ECC:  {KEP_TARGET.ecc:.6f}")
print(f"INC:  {KEP_TARGET.inc_deg:.4f} deg")
print(f"RAAN: {KEP_TARGET.raan_deg:.4f} deg")
print(f"ARGP: {KEP_TARGET.argp_deg:.4f} deg")
print(f"TA:   {KEP_TARGET.ta_deg:.4f} deg\n")
print("Chaser Kep elements:")
print(f"SMA:  {KEP_CHASER.a:.4f} m")
print(f"ECC:  {KEP_CHASER.ecc:.6f}")
print(f"INC:  {KEP_CHASER.inc_deg:.4f} deg")
print(f"RAAN: {KEP_CHASER.raan_deg:.4f} deg")
print(f"ARGP: {KEP_CHASER.argp_deg:.4f} deg")
print(f"TA:   {KEP_CHASER.ta_deg:.4f} deg\n")

#2. Compute the Keplerian orbit period for each satellite.
TP_TARGET = 2*pi*sqrt((KEP_TARGET.a**3)/KeplerianElements.mu_earth)
TP_CHASER = 2*pi*sqrt((KEP_CHASER.a**3)/KeplerianElements.mu_earth)
print(f"Target Period {TP_TARGET:.4f} s")
print(f"Chaser Period {TP_CHASER:.4f} s\n")

#3. Compute the semi-major axis difference between the target and chaser orbits.
SMA_DIFF = KEP_TARGET.a - KEP_CHASER.a
print(f"SMA diff between target and chaser: {SMA_DIFF:.4f} m\n")

#4. Use the semi-major axis difference to estimate the ΔV required to raise the Chaser to the target orbit.
DELTA_V_TO_RAISE_CHASER_TO_TARGET = SMA_DIFF*(pi/(TP_CHASER))
print(f"Delta v to raise chaser to target: {DELTA_V_TO_RAISE_CHASER_TO_TARGET:.4f} m/s\n")

#5. Compute the angular difference (phase angle) between the Chaser and Target at the initial epoch.
PHASE_DIFF = (KEP_TARGET.argp_deg + KEP_TARGET.ta_deg) - (KEP_CHASER.argp_deg + KEP_CHASER.ta_deg)
print(f"Anglular separation between target and chaser: {PHASE_DIFF:.4f} deg\n")

#6. Using the Keplerian periods from Problem 2, compute the phase angle rate between the two satellites.
PHASE_RATE_RAD_PER_SEC = 2*pi*(1/(TP_TARGET) - 1/(TP_CHASER))
PHASE_RATE_DEG_PER_DAY = degrees(PHASE_RATE_RAD_PER_SEC)*60*60*24
print(f"Phase Angle Rate: {PHASE_RATE_DEG_PER_DAY:.4f} deg/day")

Target Kep elements:
SMA:  7780000.0008 m
ECC:  0.001000
INC:  28.5000 deg
RAAN: 40.0000 deg
ARGP: 30.0000 deg
TA:   100.1128 deg

Chaser Kep elements:
SMA:  7759999.9996 m
ECC:  0.001000
INC:  28.5000 deg
RAAN: 40.0000 deg
ARGP: 30.0000 deg
TA:   85.1142 deg

Target Period 6829.3658 s
Chaser Period 6803.0484 s

SMA diff between target and chaser: 20000.0012 m

Delta v to raise chaser to target: 9.2358 m/s

Anglular separation between target and chaser: 14.9987 deg

Phase Angle Rate: -17.6187 deg/day


## Part 2: Hohmann Transfer Burn Planning

> **Assume e = 0 for the initial and final orbits (circular).**

7. Compute the semi-major axis of the intermediate orbit.
8. Compute the ΔV for each of the two burns in the Hohmann transfer sequence.
9. Compute the period of the intermediate orbit.
10. Compute the phase angle rate between the intermediate orbit and the target orbit.
11. How much will the phase angle change in the half-orbit between the burns? (Use absolute value.)
12. Subtract the answer in Problem 11 from the initial phase angle (Problem 5). This is the angular distance the Chaser must travel before the first burn.
13. Using the initial phase angle rate (Problem 6), compute how long it will take the Chaser to travel the distance from Problem 12. (Take absolute value if negative.)
14. State the date/time of the **first burn** (epoch + time from Problem 13).
15. State the date/time of the **second burn**.

In [3]:
# 7. Compute the semi-major axis of the intermediate orbit.
SMA_INT = (KEP_CHASER.a + KEP_TARGET.a)/2
print(f"Intermediate SMA: {SMA_INT:.4f} m\n")

# 8. Compute the ΔV for each of the two burns in the Hohmann transfer sequence.
DELTA_V_BURN_1 = sqrt(KeplerianElements.mu_earth*(2/KEP_CHASER.a - 1/SMA_INT)) - sqrt(KeplerianElements.mu_earth/KEP_CHASER.a)
DELTA_V_BURN_2 = sqrt(KeplerianElements.mu_earth/KEP_TARGET.a) - sqrt(KeplerianElements.mu_earth*(2/KEP_TARGET.a - 1/SMA_INT))
print(f"Delta v for first burn: {DELTA_V_BURN_1:.4f} m/s")
print(f"Delta v for second burn: {DELTA_V_BURN_2:.4f} m/s")
print(f"Total delta v: {DELTA_V_BURN_1 + DELTA_V_BURN_2:.4f} m/s\n")

# 9. Compute the period of the intermediate orbit.
INT_TP = compute_period(SMA_INT, KeplerianElements.mu_earth)
print(f"Intermediate orbit period: {INT_TP:.4f} s\n")

# 10. Compute the phase angle rate between the intermediate orbit and the target orbit.
PHASE_ANGLE_RATE_INT_TO_TARGET_RAD_PER_SEC = 2*pi*(1/(TP_TARGET) - 1/(INT_TP))
PHASE_ANGLE_RATE_INT_TO_TARGET_DEG_PER_DAY = degrees(PHASE_ANGLE_RATE_INT_TO_TARGET_RAD_PER_SEC)*60*60*24
print(f"Phase angle rate between intermediate and target orbits: {PHASE_ANGLE_RATE_INT_TO_TARGET_DEG_PER_DAY:.4f} deg/day\n")

# 11. How much will the phase angle change in the half-orbit between the burns? (Use absolute value.)
HALF_INT_ORBIT_PHASE_CHANGE = degrees(PHASE_ANGLE_RATE_INT_TO_TARGET_RAD_PER_SEC*INT_TP)/2
print(f"Phase angle change between burns: {HALF_INT_ORBIT_PHASE_CHANGE:.4f} deg\n")

# 12. Subtract the answer in Problem 11 from the initial phase angle (Problem 5). This is the angular distance the Chaser must travel before the first burn.
ANGULAR_DISTANCE_CHASER_BEFORE_BURN_1 = PHASE_DIFF+HALF_INT_ORBIT_PHASE_CHANGE
print(f"Angular distance Chaser must travel before first burn: {ANGULAR_DISTANCE_CHASER_BEFORE_BURN_1:.4f} deg\n")

# 13. Using the initial phase angle rate (Problem 6), compute how long it will take the Chaser to travel the distance from Problem 12. (Take absolute value if negative.)
SECONDS_TO_TRAVEL_TO_BURN_1 = abs(1/(PHASE_RATE_RAD_PER_SEC/radians(ANGULAR_DISTANCE_CHASER_BEFORE_BURN_1)))
print(f"Seconds to travel to point of first burn: {SECONDS_TO_TRAVEL_TO_BURN_1:.4f} s\n")

# 14. State the date/time of the **first burn** (epoch + time from Problem 13).
BURN_TIME_1 = EPOCH + timedelta(seconds=SECONDS_TO_TRAVEL_TO_BURN_1)
print(f"Burn 1 time: {str(BURN_TIME_1)}\n")

# 15. State the date/time of the **second burn**.
BURN_TIME_2 = BURN_TIME_1 + timedelta(seconds=(INT_TP/2))
print(f"Burn 2 time: {str(BURN_TIME_2)}\n")

Intermediate SMA: 7770000.0002 m

Delta v for first burn: 4.6105 m/s
Delta v for second burn: 4.6075 m/s
Total delta v: 9.2180 m/s

Intermediate orbit period: 6816.2029 s

Phase angle rate between intermediate and target orbits: -8.7952 deg/day

Phase angle change between burns: -0.3469 deg

Angular distance Chaser must travel before first burn: 14.6517 deg

Seconds to travel to point of first burn: 71850.1180 s

Burn 1 time: 2012-05-01 19:57:30.117992

Burn 2 time: 2012-05-01 20:54:18.219434



## Part 3: Ground Site Computations

| Site | Latitude (deg) | Longitude (deg) | Altitude (m) |
|------|---------------|-----------------|--------------|
| DGSA | -7.2700       | 72.3700         | -68.4        |
| VTSA | 34.8233       | 239.4983        | 269.4        |

16. Create a Chaser ephemeris spanning **5/1/2012 – 5/2/2012** using:
    - RK4 integrator, step size = 60 seconds
    - Central body-only gravity
17. Is the Chaser in view (elevation > 0°) of **DGSA or VTSA** at the **first burn**? Show computed elevations.
18. Is the Chaser in view (elevation > 0°) of **DGSA or VTSA** at the **second burn**? Show computed elevations.

In [25]:
import copy
DGSA_LAT = radians(-7.27)
DGSA_LON = radians(72.37)
DGSA_ALTD = radians(-68.4)
VTSA_LAT = radians(34.8233)
VTSA_LON = radians(239.4983)
VTSA_ALTD = radians(269.4)
R_ECEF_TOPOCENTRIC_DGSA = np.array([
    [-sin(DGSA_LON), cos(DGSA_LON), 0],
    [-sin(DGSA_LAT)*cos(DGSA_LON), -sin(DGSA_LAT)*sin(DGSA_LON), cos(DGSA_LAT)],
    [ cos(DGSA_LAT)*cos(DGSA_LON), cos(DGSA_LAT)*sin(DGSA_LON), sin(DGSA_LAT)]
])
R_ECEF_TOPOCENTRIC_VTSA = np.array([
    [-sin(VTSA_LON), cos(VTSA_LON), 0],
    [-sin(VTSA_LAT)*cos(VTSA_LON), -sin(VTSA_LAT)*sin(VTSA_LON), cos(VTSA_LAT)],
    [ cos(VTSA_LAT)*cos(VTSA_LON), cos(VTSA_LAT)*sin(VTSA_LON), sin(VTSA_LAT)]
])
Step_Size = 60
Step_Num = 1440
start_time = EPOCH
Earth_rotation = 72.921151467e-6 #rad/sec
Earth_gravitational_parameter = 3.986004418e14 #m^3/s^2
Earth_radius = 6378137 #m
f = 1/298.257223563
Earth_Eccentricity = sqrt(2*f-f**2)
class Step:
    def __init__(self, T, V1_RK_X, V1_RK_Y, V1_RK_Z, V1_RK_XD, V1_RK_YD, V1_RK_ZD,
                 V2_RK_X, V2_RK_Y, V2_RK_Z, V2_RK_XD, V2_RK_YD, V2_RK_ZD, V1_AZ_DGSA, 
                 V1_EL_DGSA, V2_AZ_DGSA, V2_EL_DGSA, V1_AZ_VTSA, V1_EL_VTSA, V2_AZ_VTSA, V2_EL_VTSA, SepA):
        self.T = T
        self.V1_RK_X = V1_RK_X
        self.V1_RK_Y = V1_RK_Y
        self.V1_RK_Z = V1_RK_Z
        self.V1_RK_XD = V1_RK_XD
        self.V1_RK_YD = V1_RK_YD
        self.V1_RK_ZD = V1_RK_ZD
        self.V2_RK_X = V2_RK_X
        self.V2_RK_Y = V2_RK_Y
        self.V2_RK_Z = V2_RK_Z
        self.V2_RK_XD = V2_RK_XD
        self.V2_RK_YD = V2_RK_YD
        self.V2_RK_ZD = V2_RK_ZD
        self.V1_AZ_DGSA = V1_AZ_DGSA
        self.V1_EL_DGSA = V1_EL_DGSA
        self.V2_AZ_DGSA = V2_AZ_DGSA
        self.V2_EL_DGSA = V2_EL_DGSA
        self.V1_AZ_VTSA = V1_AZ_VTSA
        self.V1_EL_VTSA = V1_EL_VTSA
        self.V2_AZ_VTSA = V2_AZ_VTSA
        self.V2_EL_VTSA = V2_EL_VTSA
        self.SepA = SepA

steps_central_body:list[Step] = []
V1_y_0 = Vector6(POS_TARGET.x, POS_TARGET.y, POS_TARGET.z, VEL_TARGET.x, VEL_TARGET.y, VEL_TARGET.z)
V2_y_0 = Vector6(POS_CHASER.x, POS_CHASER.y, POS_CHASER.z, VEL_CHASER.x, VEL_CHASER.y, VEL_CHASER.z)
h = Step_Size
w_ = Vector3(0, 0, 72.921151467e-6)
V1_v_r_ = VEL_TARGET - (w_.cross(POS_TARGET))
V2_v_r_ = VEL_CHASER - (w_.cross(POS_CHASER))
V1_temp_pos = copy.deepcopy(POS_TARGET)
V2_temp_pos = copy.deepcopy(POS_CHASER)
V1_temp_vel = copy.deepcopy(VEL_TARGET)
V2_temp_vel = copy.deepcopy(VEL_CHASER)
timestamp = start_time

# calculate ECEF position vectors
V1_gah, V1_pos_ECEF_np = rotate_TOD_to_ECEF(start_time, POS_TARGET, Earth_rotation)
V2_gah, V2_pos_ECEF_np = rotate_TOD_to_ECEF(start_time, POS_CHASER, Earth_rotation)
V1_pos_ECEF = Vector3(V1_pos_ECEF_np[0][0], V1_pos_ECEF_np[1][0], V1_pos_ECEF_np[2][0])
V2_pos_ECEF = Vector3(V2_pos_ECEF_np[0][0], V2_pos_ECEF_np[1][0], V2_pos_ECEF_np[2][0])

# calculate az/el DGSA:
V1_lat, V1_lon, V1_height = calculate_lat_lon_h(Earth_Eccentricity, V1_pos_ECEF, Earth_radius)
V2_lat, V2_lon, V2_height = calculate_lat_lon_h(Earth_Eccentricity, V2_pos_ECEF, Earth_radius)
V1_TOCOCENTRIC_Pos_sat = R_ECEF_TOPOCENTRIC_DGSA @ V1_pos_ECEF_np
V2_TOCOCENTRIC_Pos_sat = R_ECEF_TOPOCENTRIC_DGSA @ V2_pos_ECEF_np
Sen_ECEF = convert_lat_lon_h_to_ecef(DGSA_LAT, DGSA_LON, DGSA_ALTD, Earth_radius)
Sen_TOCOCENTRIC = R_ECEF_TOPOCENTRIC_DGSA @ Sen_ECEF.get_np_vector()
sen_to_v1 = V1_TOCOCENTRIC_Pos_sat - Sen_TOCOCENTRIC
sen_to_v2 = V2_TOCOCENTRIC_Pos_sat - Sen_TOCOCENTRIC
V1_Az_DGSA = degrees(atan2(sen_to_v1[0][0], sen_to_v1[1][0]))% 360.0
V1_El_DGSA = degrees(atan(sen_to_v1[2][0]/sqrt(sen_to_v1[0][0]**2 + sen_to_v1[1][0]**2)))
V2_Az_DGSA = degrees(atan2(sen_to_v2[0][0], sen_to_v2[1][0]))% 360.0
V2_El_DGSA = degrees(atan(sen_to_v2[2][0]/sqrt(sen_to_v2[0][0]**2 + sen_to_v2[1][0]**2)))

# calculate az/el VTSA:
V1_lat, V1_lon, V1_height = calculate_lat_lon_h(Earth_Eccentricity, V1_pos_ECEF, Earth_radius)
V2_lat, V2_lon, V2_height = calculate_lat_lon_h(Earth_Eccentricity, V2_pos_ECEF, Earth_radius)
V1_TOCOCENTRIC_Pos_sat = R_ECEF_TOPOCENTRIC_VTSA @ V1_pos_ECEF_np
V2_TOCOCENTRIC_Pos_sat = R_ECEF_TOPOCENTRIC_VTSA @ V2_pos_ECEF_np
Sen_ECEF = convert_lat_lon_h_to_ecef(VTSA_LAT, VTSA_LON, VTSA_ALTD, Earth_radius)
Sen_TOCOCENTRIC = R_ECEF_TOPOCENTRIC_VTSA @ Sen_ECEF.get_np_vector()
sen_to_v1 = V1_TOCOCENTRIC_Pos_sat - Sen_TOCOCENTRIC
sen_to_v2 = V2_TOCOCENTRIC_Pos_sat - Sen_TOCOCENTRIC
V1_Az_VTSA = degrees(atan2(sen_to_v1[0][0], sen_to_v1[1][0]))% 360.0
V1_El_VTSA = degrees(atan(sen_to_v1[2][0]/sqrt(sen_to_v1[0][0]**2 + sen_to_v1[1][0]**2)))
V2_Az_VTSA = degrees(atan2(sen_to_v2[0][0], sen_to_v2[1][0]))% 360.0
V2_El_VTSA = degrees(atan(sen_to_v2[2][0]/sqrt(sen_to_v2[0][0]**2 + sen_to_v2[1][0]**2)))

# calculate SepA
sen_to_v1_vector3 = Vector3(sen_to_v1[0][0], sen_to_v1[1][0], sen_to_v1[2][0])
sen_to_v2_vector3 = Vector3(sen_to_v2[0][0], sen_to_v2[1][0], sen_to_v2[2][0])
SepA = degrees(acos(sen_to_v1_vector3.dot(sen_to_v2_vector3)/(sen_to_v1_vector3.magnitude()*sen_to_v2_vector3.magnitude())))

steps_central_body.append(Step(0, V1_y_0.x, V1_y_0.y, V1_y_0.z, V1_temp_vel.x, V1_temp_vel.y, V1_temp_vel.z,
V2_y_0.x, V2_y_0.y, V2_y_0.z, V2_temp_vel.x, V2_temp_vel.y, V2_temp_vel.z, V1_Az_DGSA, V1_El_DGSA, V2_Az_DGSA, V2_El_DGSA,
V1_Az_VTSA, V1_El_VTSA, V2_Az_VTSA, V2_El_VTSA, SepA))

for i in range(Step_Num):
    # calculate TOD position vectors
    t = h*(i+1)
    timestamp = timestamp + timedelta(seconds=h)
    V1_step = compute_rk_2_body_step(0, Earth_gravitational_parameter, V1_temp_pos, V1_temp_vel, h, V1_y_0, timestamp, 0, 0, 0)
    V2_step = compute_rk_2_body_step(0, Earth_gravitational_parameter, V2_temp_pos, V2_temp_vel, h, V2_y_0, timestamp, 0, 0, 0)
    
     # calculate ECEF position vectors
    V1_gah, V1_pos_ECEF_np = rotate_TOD_to_ECEF(timestamp, Vector3(V1_step.x, V1_step.y, V1_step.z), Earth_rotation)
    V2_gah, V2_pos_ECEF_np = rotate_TOD_to_ECEF(timestamp, Vector3(V2_step.x, V2_step.y, V2_step.z), Earth_rotation)
    SEN_gah, SEN_pos_ECEF_np = rotate_TOD_to_ECEF(timestamp, Vector3(V2_step.x, V2_step.y, V2_step.z), Earth_rotation)
    V1_pos_ECEF = Vector3(V1_pos_ECEF_np[0][0], V1_pos_ECEF_np[1][0], V1_pos_ECEF_np[2][0])
    V2_pos_ECEF = Vector3(V2_pos_ECEF_np[0][0], V2_pos_ECEF_np[1][0], V2_pos_ECEF_np[2][0])
    
    # calculate az/el DGSA:
    V1_lat, V1_lon, V1_height = calculate_lat_lon_h(Earth_Eccentricity, V1_pos_ECEF, Earth_radius)
    V2_lat, V2_lon, V2_height = calculate_lat_lon_h(Earth_Eccentricity, V2_pos_ECEF, Earth_radius)
    V1_TOCOCENTRIC_Pos_sat = R_ECEF_TOPOCENTRIC_DGSA @ V1_pos_ECEF_np
    V2_TOCOCENTRIC_Pos_sat = R_ECEF_TOPOCENTRIC_DGSA @ V2_pos_ECEF_np
    Sen_ECEF = convert_lat_lon_h_to_ecef(DGSA_LAT, DGSA_LON, DGSA_ALTD, Earth_radius)
    Sen_TOCOCENTRIC = R_ECEF_TOPOCENTRIC_DGSA @ Sen_ECEF.get_np_vector()
    sen_to_v1 = V1_TOCOCENTRIC_Pos_sat - Sen_TOCOCENTRIC
    sen_to_v2 = V2_TOCOCENTRIC_Pos_sat - Sen_TOCOCENTRIC
    V1_Az_DGSA = degrees(atan2(sen_to_v1[0][0], sen_to_v1[1][0]))% 360.0
    V1_El_DGSA = degrees(atan(sen_to_v1[2][0]/sqrt(sen_to_v1[0][0]**2 + sen_to_v1[1][0]**2)))
    V2_Az_DGSA = degrees(atan2(sen_to_v2[0][0], sen_to_v2[1][0]))% 360.0
    V2_El_DGSA = degrees(atan(sen_to_v2[2][0]/sqrt(sen_to_v2[0][0]**2 + sen_to_v2[1][0]**2)))

    # calculate az/el VTSA:
    V1_lat, V1_lon, V1_height = calculate_lat_lon_h(Earth_Eccentricity, V1_pos_ECEF, Earth_radius)
    V2_lat, V2_lon, V2_height = calculate_lat_lon_h(Earth_Eccentricity, V2_pos_ECEF, Earth_radius)
    V1_TOCOCENTRIC_Pos_sat = R_ECEF_TOPOCENTRIC_VTSA @ V1_pos_ECEF_np
    V2_TOCOCENTRIC_Pos_sat = R_ECEF_TOPOCENTRIC_VTSA @ V2_pos_ECEF_np
    Sen_ECEF = convert_lat_lon_h_to_ecef(VTSA_LAT, VTSA_LON, VTSA_ALTD, Earth_radius)
    Sen_TOCOCENTRIC = R_ECEF_TOPOCENTRIC_VTSA @ Sen_ECEF.get_np_vector()
    sen_to_v1 = V1_TOCOCENTRIC_Pos_sat - Sen_TOCOCENTRIC
    sen_to_v2 = V2_TOCOCENTRIC_Pos_sat - Sen_TOCOCENTRIC
    V1_Az_VTSA = degrees(atan2(sen_to_v1[0][0], sen_to_v1[1][0]))% 360.0
    V1_El_VTSA = degrees(atan(sen_to_v1[2][0]/sqrt(sen_to_v1[0][0]**2 + sen_to_v1[1][0]**2)))
    V2_Az_VTSA = degrees(atan2(sen_to_v2[0][0], sen_to_v2[1][0]))% 360.0
    V2_El_VTSA = degrees(atan(sen_to_v2[2][0]/sqrt(sen_to_v2[0][0]**2 + sen_to_v2[1][0]**2)))

    # calculate sepA
    sen_to_v1_vector3 = Vector3(sen_to_v1[0][0], sen_to_v1[1][0], sen_to_v1[2][0])
    sen_to_v2_vector3 = Vector3(sen_to_v2[0][0], sen_to_v2[1][0], sen_to_v2[2][0])
    SepA = degrees(acos(sen_to_v1_vector3.dot(sen_to_v2_vector3)/(sen_to_v1_vector3.magnitude()*sen_to_v2_vector3.magnitude())))

    # append data to array and move on
    steps_central_body.append(Step(t, V1_step.x, V1_step.y, V1_step.z, V1_step.xd, V1_step.yd, V1_step.zd,
        V2_step.x, V2_step.y, V2_step.z, V2_step.xd, V2_step.yd, V2_step.zd, V1_Az_DGSA, V1_El_DGSA, V2_Az_DGSA, V2_El_DGSA,
        V1_Az_VTSA, V1_El_VTSA, V2_Az_VTSA, V2_El_VTSA, SepA))
    V1_y_0 = V1_step
    V2_y_0 = V2_step
    V1_temp_pos = Vector3(V1_step.x, V1_step.y, V1_step.z)
    V1_temp_vel = Vector3(V1_step.xd, V1_step.yd, V1_step.zd)
    V2_temp_pos = Vector3(V2_step.x, V2_step.y, V2_step.z)
    V2_temp_vel = Vector3(V2_step.xd, V2_step.yd, V2_step.zd)

from IPython.display import Markdown, display

header = "| Time (s) | T (seconds) | CHASER_X (m) | CHASER_Y (m) | CHASER_Z (m) | CHASER_XD (m/s) | CHASER_YD (m/s) | CHASER_ZD (m/s) | CHASER_AZ_DGSA | CHASER_EL_DGSA | CHASER_AZ_VTSA | CHASER_EL_VTSA |"
separator = "|------|-----------------|----------|----------|----------|--------------|--------------|--------------|-----------|-----------|---------|---------|"

rows = []
for i, s in enumerate(steps_central_body):
    row = (
        f"{EPOCH+timedelta(seconds=s.T)} | {s.T}"
        f"| {s.V2_RK_X:.2f} | {s.V2_RK_Y:.2f} | {s.V2_RK_Z:.2f} | {s.V2_RK_XD:.2f} | {s.V2_RK_YD:.2f} | {s.V2_RK_ZD:.2f} | {s.V2_AZ_DGSA:.3f} | {s.V2_EL_DGSA:.3f} | {s.V2_AZ_VTSA:.3f} | {s.V2_EL_VTSA:.3f} |"
    )
    rows.append(row)

table_md = "\n".join([header, separator] + rows)

# display(Markdown(f"{table_md}"))

# 2012-05-01 19:57:00	71820	7309069.83	-113394.58	-2598067.35	963.14	6672.13	2438.99	118.413	-57.692	221.317	-24.598
# 2012-05-01 19:58:00	71880	7355599.24	286902.08	-2447811.69	587.44	6667.68	2568.26	118.367	-59.358	219.563	-22.937

# 2012-05-01 20:54:00	75240	-7333463.16	-37639.88	2543761.26	-834.22	-6669.11	-2482.72	293.438	-26.995	51.600	-48.890
# 2012-05-01 20:55:00	75300	-7372244.02	-437523.91	2390972.94	-458.15	-6656.95	-2608.92	293.009	-25.085	50.345	-50.441

print("At the first burn  (2012-05-01 19:57:30.117992), DGSA EL is between -57.692 deg and -59.358 deg and the VTSA EL is between -24.598 deg and -22.937 deg so NEITHER sensor has visibility of chaser")
print("At the second burn (2012-05-01 20:54:18.219434), DGSA EL is between -26.995 deg and -25.085 deg and the VTSA EL is between -48.890 deg and -50.441 deg so NEITHER sensor has visibility of chaser")

At the first burn  (2012-05-01 19:57:30.117992), DGSA EL is between -57.692 deg and -59.358 deg and the VTSA EL is between -24.598 deg and -22.937 deg so NEITHER sensor has visibility of chaser
At the second burn (2012-05-01 20:54:18.219434), DGSA EL is between -26.995 deg and -25.085 deg and the VTSA EL is between -48.890 deg and -50.441 deg so NEITHER sensor has visibility of chaser


## Part 4: Eclipse Computations

> Use UTC time as input to sun computations.

19. How long is an eclipse for the **target orbit** and for the **initial chaser orbit**?
20. State whether each burn occurs in **sunlight or eclipse**. Show eclipse start/stop times relative to each burn.

## Extra Credit (20 points)

Create an ephemeris for the chaser satellite including impulsive burns, and plot the **angular separation** between chaser and target as a function of time to demonstrate successful rendezvous.

---

## Check Your Work

### Chaser Ephemeris Spot Check

| Parameter | Value | Units |
|-----------|-------|-------|
| Time | 01-May-12 03:06:00.0 | |
| T | 11160 | seconds |
| X | 6848848.185 | m |
| Y | 3516274.124 | m |
| Z | -927766.0507 | m |
| Xdot | -2548.884238 | m/s |
| Ydot | 5827.795778 | m/s |
| Zdot | 3313.518432 | m/s |
| Azimuth DGSA | 92.811 | deg |
| Elevation DGSA | -12.017 | deg |
| Azimuth VTSA | 281.186 | deg |
| Elevation VTSA | -55.065 | deg |
| Sun-Vehicle-Earth angle | 155.161 | deg |
| In-Sun (0=eclipse, 1=sun) | 1 | |
| Phase Angle | 12.744 | deg |

### Intermediate Orbit Check

$r_{p,\text{intermed}} = 7{,}760{,}000 \text{ m}$

$r_{a,\text{intermed}} = 7{,}780{,}000 \text{ m}$

> Time from initial epoch to **second burn** is less than one day.